# Encode the corpus with bge-m3 (Kaggle GPU)

Produces `embeddings.npy` (float16, L2-normalized), `doc_ids.json` and `meta.json` for R2.

**Setup**
1. Upload `data/processed/corpus.parquet` as a private Kaggle Dataset and add it to this notebook.
2. Settings: Accelerator = **GPU T4 x2** (one GPU is used), Internet = **on**.
3. Run all. Then download the `dense/` folder from the Output tab into `artifacts/dense/`.
4. Locally: `uv run python scripts/verify_dense.py`.

`MODEL`, `REVISION` and `MAX_SEQ_LENGTH` must match `configs/r2.yaml`; the sentence-transformers
version matches `uv.lock`. `verify_dense.py` re-encodes a sample locally to prove it.

In [ ]:
!pip install -q "sentence-transformers==6.2.0"

In [ ]:
import glob
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sentence_transformers
import torch
import transformers
from sentence_transformers import SentenceTransformer

MODEL = "BAAI/bge-m3"
REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 16  # lower to 8 if CUDA runs out of memory (longest texts are encoded first)
OUT_DIR = Path("/kaggle/working/dense")

assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook settings"
print(torch.cuda.get_device_name(0), "| sentence-transformers", sentence_transformers.__version__)

In [ ]:
(corpus_path,) = glob.glob("/kaggle/input/**/corpus.parquet", recursive=True)
corpus = pd.read_parquet(corpus_path)
assert corpus["doc_id"].is_unique
print(f"{len(corpus):,} articles from {corpus_path}")

In [ ]:
model = SentenceTransformer(MODEL, revision=REVISION, device="cuda")
model.half()  # fp16 on GPU: ~2x faster, vectors are stored as fp16 anyway
model.max_seq_length = MAX_SEQ_LENGTH
print(model)

In [ ]:
start = time.perf_counter()
embeddings = model.encode(
    corpus["contents"].tolist(),
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float16)
seconds = time.perf_counter() - start

norms = np.linalg.norm(embeddings.astype(np.float32), axis=1)
assert np.isfinite(embeddings).all(), "NaN/inf in embeddings"
print(f"{embeddings.shape} in {seconds / 60:.1f} min | norms {norms.min():.4f}-{norms.max():.4f}")

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
np.save(OUT_DIR / "embeddings.npy", embeddings)
(OUT_DIR / "doc_ids.json").write_text(json.dumps(corpus["doc_id"].tolist()))
meta = {
    "model": MODEL,
    "revision": REVISION,
    "max_seq_length": MAX_SEQ_LENGTH,
    "normalized": True,
    "dtype": "float16",
    "n_docs": len(corpus),
    "encode_seconds": seconds,
    "gpu": torch.cuda.get_device_name(0),
    "versions": {
        "sentence_transformers": sentence_transformers.__version__,
        "transformers": transformers.__version__,
        "torch": torch.__version__,
    },
}
(OUT_DIR / "meta.json").write_text(json.dumps(meta, indent=2))
for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name}: {f.stat().st_size / 1e6:.1f} MB")